In [ ]:
import os

print(os.environ.get("CONDA_DEFAULT_ENV"))

In [ ]:
# %load init_ipynb
import os
import sys
import matplotlib.pyplot as pl

sys.path.insert(0, '..')
# assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {'axes.labelsize': 16,
           'font.size': 16,
           'legend.fontsize': 16,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           #'text.usetex': False,
           'xtick.direction': 'in',
           'ytick.direction': 'in'
           }
pl.rcParams.update(plotpar)


In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="CastroGinard2022", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1)
df = tabs[0].to_pandas()
df_mem = tabs[1].to_pandas()
df_mem.head()

In [ ]:
df_young = df[df['logAge']<7.5]

In [ ]:
from utils import plot_bls
import matplotlib.pyplot as pl
import lightkurve as lk
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

author = 'QLP'
cols = ['RA_ICRS','DE_ICRS']
for name in tqdm(df_young.Cluster.values):

    cluster = df_mem.query("Cluster==@name")
    pos = cluster[cols].values[0]
    res_all = lk.search_lightcurve(f"{pos[0]} {pos[1]}", author=author, radius=60)
    if len(res_all)>0:
                
        outdir = Path(cd.catalog_name,name)
        outdir.mkdir(parents=True, exist_ok=True)

        for i,row in tqdm(cluster.iterrows(), desc=name):
            pos = row[cols].values
            res = lk.search_lightcurve(f"{pos[0]} {pos[1]}", author=author)
            if len(res)>1:
                lc = res[0].download().normalize()
                texp = res[0].exptime.value[0] / 60.0 / 24.0

                ax = lc.scatter(label=f"TIC {lc.TICID}")
                pl.close()

                x = np.ascontiguousarray(lc.time.value, dtype=np.float64)
                y = np.ascontiguousarray(lc.flux.value, dtype=np.float64)
                fig = plot_bls(x, y)
                fp = outdir.joinpath(f"{lc.TICID}.png")
                fig.savefig(fp)
                print("Saved: ", fp)
                pl.close()